In [5]:
import pyqcu

In [6]:
from pyqcu import cuda

In [7]:
cuda.argv

tensor([-3.5000e+00,  1.0000e-09,  1.0000e-01])

In [8]:
cuda.params

tensor([     32,      32,      32,      32, 1048576,       1,       1,       1,
              1,       0,       0,       1,       0,    1000,       2,       0,
              0,       4,       4,       4,       8,      24,       1,      42],
       dtype=torch.int32)

In [11]:
cuda.set_ptrs.dtype

torch.int64

In [1]:
import pyquda

cannot import name 'pyqcu' from partially initialized module 'pyquda' (most likely due to a circular import) (/root/lib/PyQuda-master/pyquda/__init__.py)


In [1]:
from pyqcu.cuda import params, argv, set_ptrs
from pyqcu import cuda
from pyquda.utils import gauge_utils
import mpi4py.MPI as MPI
from pyquda.field import LatticeFermion
from pyquda import core, quda, mpi
from pyqcu.cuda import qcu
from time import perf_counter
import cupy as cp
import torch
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
Nd, Ns, Nc = 4, 4, 3
latt_size = [8, 16, 16, 16]
grid_size = [1, 1, 1, 1]
Lx, Ly, Lz, Lt = latt_size
Gx, Gy, Gz, Gt = grid_size
latt_size = [Lx // Gx, Ly // Gy, Lz // Gz, Lt // Gt]
Lx, Ly, Lz, Lt = latt_size
Vol = Lx * Ly * Lz * Lt
xi_0, nu = 1, 1
mass = 0
coeff_r, coeff_t = 0, 0
mpi.init(grid_size)
print(f"single latt size = {latt_size}")
p = LatticeFermion(latt_size, cp.ones(
    [Lt, Lz, Ly, Lx, Ns, Nc * 2]).view(cp.complex128))
qcu_p = LatticeFermion(latt_size)
quda_p = LatticeFermion(latt_size)
qcu_x = LatticeFermion(latt_size)
quda_x = LatticeFermion(latt_size)
dslash = core.getDslash(
    latt_size,
    mass,
    1e-9,
    1000,
    xi_0,
    nu,
    coeff_t,
    coeff_r,
    multigrid=False,
    anti_periodic_t=False,
)
U = gauge_utils.gaussGauge(latt_size, 0)
dslash.loadGauge(U)



/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:

params[cuda._LAT_X_] = 16
params[cuda._LAT_Y_] = 16
params[cuda._LAT_Z_] = 16
params[cuda._LAT_T_] = 16
params[cuda._LAT_XYZT_] = params[cuda._LAT_X_] * \
    params[cuda._LAT_Y_]*params[cuda._LAT_Z_]*params[cuda._LAT_T_]
params[cuda._DATA_TYPE_] = cuda._LAT_C128_
params[cuda._SET_PLAN_] = 1
print("Parameters:", params)
argv[cuda._MASS_] = mass
print("Arguments:", argv)
qcu.applyInitQcu(set_ptrs, params, argv)


In [ ]:


def compare(round):
    # quda
    cp.cuda.runtime.deviceSynchronize()
    if rank == 0:
        print("================quda=================")
    t1 = perf_counter()
    quda.invertQuda(quda_x.data_ptr, p.data_ptr, dslash.invert_param)
    # D*x=p, to get quda_x
    cp.cuda.runtime.deviceSynchronize()
    t2 = perf_counter()
    quda.MatQuda(quda_p.data_ptr, quda_x.data_ptr, dslash.invert_param)
    # quda_p=D*quda_x
    cp.cuda.runtime.deviceSynchronize()
    print(
        f"rank {rank} quda x and x difference: , {cp.linalg.norm(quda_p.data - p.data) / cp.linalg.norm(quda_p.data)}, takes {t2 - t1} sec, norm_quda_x = {cp.linalg.norm(quda_x.data)}"
    )
    print(f"quda rank {rank} takes {t2 - t1} sec")
    # qcu
    _U = torch.Tensor(U.data.numpy()).cuda()
    _p = torch.Tensor(p.data.numpy()).cuda()
    _qcu_x = torch.Tensor(qcu_x.data.numpy()).cuda()
    cp.cuda.runtime.deviceSynchronize()
    if rank == 0:
        print("===============qcu==================")
    t1 = perf_counter()
    qcu.applyWilsonBistabCgQcu(_qcu_x, _p, _U, set_ptrs, params)
    qcu_x.data = cp.array(_qcu_x.cpu().numpy())
    # D*x=p, to get qcu_x
    cp.cuda.runtime.deviceSynchronize()
    t2 = perf_counter()
    quda.MatQuda(qcu_p.data_ptr, qcu_x.data_ptr, dslash.invert_param)
    # qcu_p=D*qcu_x
    print(
        f"rank {rank} my x and x difference: , {cp.linalg.norm(qcu_p.data - p.data) / cp.linalg.norm(qcu_p.data)}, takes {t2 - t1} sec, my_x_norm = {cp.linalg.norm(qcu_x.data)}"
    )
    print(f"qcu rank {rank} takes {t2 - t1} sec")
    print("============================")


for i in range(0, 10):
    compare(i)
qcu.applyEndQcu(set_ptrs, params)